# Fixed-Depth vs Adaptive Routing with Imported LCI

**Authors**: Romain Sacchi (PSI)

**Contact**: romain.sacchi@psi.ch

## Purpose

This notebook follows the same workflow as Notebook 2.2, but compares two ways
of deciding how far TRAILS should explicitly route temporal technosphere
branches:

1. **Fixed-depth routing**: route every branch up to an absolute `max_depth`.
2. **Adaptive routing**: route branches until their estimated score potential is
   smaller than a cutoff.

The example uses a real premise-style TRAILS datapackage and imports the
foreground passenger-car inventory `lci-pass_cars.xlsx`. The same foreground
activity, demand amount, year, and LCIA method are used in every case.

## What to look for

The comparison table at the end reports:

- number of routed nodes and edges;
- maximum graph depth reached;
- routing and LCA runtime;
- total score;
- score difference relative to the deepest fixed-depth case in this notebook.

Adaptive routing does **not** discard branches below the cutoff. Those branches
remain frontier demands and are still included in the matrix solve during
`trails.lca(...)`.

In [ ]:
import logging
import time
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.express as px
from datapackage import Package

from trails import Trails, plot_temporal_scores, search_activity
from trails.logging import configure_trails_logging
from trails.plotting import plot_temporal_graph

## 1. Choose the data package and imported LCI file

The datapackage zip is large and is not committed to the repository. The default
path below points to the local case-study datapackage used during development.
Edit `datapackage_path` if your zip is elsewhere.

The imported foreground inventory is the tracked example workbook used in
Notebook 2.2.

In [ ]:
# If you run this notebook from the examples/ folder, this relative path works.
datapackage_path = Path("../dev/trails_remind_SSP2-PkBudg1000.zip")

# If you run this notebook from the repository root, use this fallback.
if not datapackage_path.exists():
    datapackage_path = Path("dev/trails_remind_SSP2-PkBudg1000.zip")

inventory_path = Path("lci-pass_cars.xlsx")
if not inventory_path.exists():
    inventory_path = Path("examples") / "lci-pass_cars.xlsx"

if not datapackage_path.exists():
    raise FileNotFoundError(
        "Set `datapackage_path` to a real TRAILS datapackage zip before running."
    )
if not inventory_path.exists():
    raise FileNotFoundError("Could not find `lci-pass_cars.xlsx`.")

print("Datapackage:", datapackage_path.resolve())
print("Imported LCI:", inventory_path.resolve())

## 2. Set the assessment choices

The LCIA method is configured once when the `Trails` object is created. The same
method is then used for adaptive routing and for the final LCA score.

The settings below are intentionally modest so the notebook remains practical to
run. Increase `fixed_depth_2` to `3` or add more adaptive cutoff values if you
want a stricter convergence check.

In [ ]:
ei_version = "3.12"
methods = [
    "EF v3.1 - climate change - global warming potential (GWP100)",
]

ref_year = 2050
demand_amount = 1.0
min_amount = 1e-12
show_progress = False

fixed_depth_1 = 1
fixed_depth_2 = 2
adaptive_relative_cutoff_1 = 1e-1
adaptive_relative_cutoff_2 = 1e-2
adaptive_min_depth = 1

run_lca = True
store_inventory = False

output_dir = inventory_path.parent / "notebook_outputs" / "routing_depth_comparison"
output_dir.mkdir(parents=True, exist_ok=True)
output_dir

## 3. Load the data package

Annual interpolation is enabled. If this is the first time the datapackage is
loaded, TRAILS will build an interpolation cache. Later runs should load from
that cache.

In [ ]:
configure_trails_logging(
    file_level=logging.INFO,
    filename=str(output_dir / "routing_depth_comparison.log"),
    also_console=False,
)

package = Package(str(datapackage_path))
trails = Trails(
    package=package,
    interpolate_annual=True,
    methods=methods,
    ei_version=ei_version,
)

print("Template scenario years:", trails.template_labels[:5], "...", trails.template_labels[-5:])
print("Interpolated year range:", trails.min_year, "to", trails.max_year)
print("A shape before import:", trails.A.shape)
print("B shape before import:", trails.B.shape)

## 4. Import the Excel foreground inventory

The import adds the foreground passenger-car activities to the loaded
datapackage. If any technosphere exchanges cannot be linked, TRAILS raises an
error instead of silently importing an incomplete inventory.

In [ ]:
trails.import_excel_inventory(str(inventory_path))

print("A shape after import:", trails.A.shape)
print("B shape after import:", trails.B.shape)

## 5. Select the imported foreground activity by metadata

Do not hard-code the activity index. The integer index can change if the
datapackage or imported inventory changes. The metadata fields are the stable
selection criteria.

In [ ]:
activity_name = "transport, passenger, car, diesel"
reference_product = "transport, passenger, car"
location = "RER"

matches = search_activity(trails, activity_name)
matches_df = pd.DataFrame(matches.rows, columns=matches.field_names)

exact_matches = matches_df[
    (matches_df["name"] == activity_name)
    & (matches_df["reference product"] == reference_product)
    & (matches_df["location"] == location)
].copy()

if len(exact_matches) != 1:
    raise ValueError(
        "Expected exactly one imported activity. Inspect `matches_df` and refine "
        "activity_name, reference_product, or location."
    )

activity_index = int(exact_matches["index"].item())
print("Selected activity index:", activity_index)
exact_matches

## 6. Inspect the imported activity

This is the same inspection step as in Notebook 2.2. It is useful before running
routing because temporal exchange settings determine when child demands appear.

In [ ]:
trails.print_exchange_table(year=ref_year, act_idx=activity_index)

## 7. Prepare a simple result table

The only helper below records the same summary after each run. Each routing case
is still run explicitly in its own cell, so it is easy to see exactly what
changes between fixed-depth and adaptive routing.

In [ ]:
results = []
score_snapshots = {}


def total_scores_from_current_run() -> dict[str, float]:
    if trails.scores is None:
        return {}

    score = trails.scores
    for dim in list(score.dims):
        if dim != "method":
            score = score.sum(dim=dim)

    return {
        str(method): float(score.sel(method=method).item())
        for method in score.coords["method"].values
    }


def add_current_run_to_results(case_name: str, mode: str, routing_time_s: float, lca_time_s: float) -> None:
    graph = trails.graph
    if graph is None:
        raise RuntimeError("No routing graph is available.")

    depths = [int(data.get("depth", 0)) for _, data in graph.nodes(data=True)]

    row = {
        "case": case_name,
        "mode": mode,
        "nodes": graph.number_of_nodes(),
        "edges": graph.number_of_edges(),
        "max_routed_depth": max(depths) if depths else 0,
        "routing_time_s": routing_time_s,
        "lca_time_s": lca_time_s,
    }

    for method, score in total_scores_from_current_run().items():
        row[f"score: {method}"] = score

    results.append(row)

    if trails.scores is not None:
        score_snapshots[case_name] = trails.scores.copy(deep=True)

## 8. Fixed-depth routing: depth 1

At depth 1, TRAILS explicitly expands only the immediate temporal child demands
of the foreground activity. Everything beyond that becomes frontier demand for
the matrix solve.

In [ ]:
case_name = "fixed depth 1"

start = time.perf_counter()
trails.temporal_routing(
    start_year=ref_year,
    start_act_idx=activity_index,
    amount=demand_amount,
    max_depth=fixed_depth_1,
    min_amount=min_amount,
    show_progress=show_progress,
    attribute_to_roots=True,
)
routing_time_s = time.perf_counter() - start

lca_time_s = np.nan
if run_lca:
    start = time.perf_counter()
    trails.lca(
        show_progress=show_progress,
        compute_score=True,
        store_inventory=store_inventory,
    )
    lca_time_s = time.perf_counter() - start

add_current_run_to_results(case_name, "fixed depth", routing_time_s, lca_time_s)
results[-1]

## 9. Fixed-depth routing: depth 2

This is the reference case used later for the score-difference table. It is not
claimed to be fully converged; it is simply the deepest fixed-depth case run in
this notebook by default.

In [ ]:
case_name = "fixed depth 2"

start = time.perf_counter()
trails.temporal_routing(
    start_year=ref_year,
    start_act_idx=activity_index,
    amount=demand_amount,
    max_depth=fixed_depth_2,
    min_amount=min_amount,
    show_progress=show_progress,
    attribute_to_roots=True,
)
routing_time_s = time.perf_counter() - start

lca_time_s = np.nan
if run_lca:
    start = time.perf_counter()
    trails.lca(
        show_progress=show_progress,
        compute_score=True,
        store_inventory=store_inventory,
    )
    lca_time_s = time.perf_counter() - start

add_current_run_to_results(case_name, "fixed depth", routing_time_s, lca_time_s)
results[-1]

## 10. Adaptive routing: relative cutoff 1

Adaptive routing has no hard depth cap here (`max_depth=None`). The relative
cutoff is multiplied by the functional unit's static score potential.

With `adaptive_relative_cutoff_1 = 1e-1`, branches are stopped once their
estimated score potential is at most 10% of the functional unit potential, after the minimum
depth has been reached.

The first adaptive run may spend extra time building the static activity score
cache. The next adaptive run can reuse it.

In [ ]:
case_name = "adaptive 1e-1"

start = time.perf_counter()
trails.temporal_routing(
    start_year=ref_year,
    start_act_idx=activity_index,
    amount=demand_amount,
    max_depth=None,
    min_amount=min_amount,
    show_progress=show_progress,
    attribute_to_roots=True,
    adaptive_relative_score_cutoff=adaptive_relative_cutoff_1,
    adaptive_min_depth=adaptive_min_depth,
    adaptive_use_cache=True,
)
routing_time_s = time.perf_counter() - start

lca_time_s = np.nan
if run_lca:
    start = time.perf_counter()
    trails.lca(
        show_progress=show_progress,
        compute_score=True,
        store_inventory=store_inventory,
    )
    lca_time_s = time.perf_counter() - start

add_current_run_to_results(case_name, "adaptive", routing_time_s, lca_time_s)
results[-1]

## 11. Adaptive routing: relative cutoff 2

A smaller relative cutoff is stricter: more branches remain eligible for
explicit routing, and the graph can become deeper or larger.

In [ ]:
case_name = "adaptive 1e-2"

start = time.perf_counter()
trails.temporal_routing(
    start_year=ref_year,
    start_act_idx=activity_index,
    amount=demand_amount,
    max_depth=None,
    min_amount=min_amount,
    show_progress=show_progress,
    attribute_to_roots=True,
    adaptive_relative_score_cutoff=adaptive_relative_cutoff_2,
    adaptive_min_depth=adaptive_min_depth,
    adaptive_use_cache=True,
)
routing_time_s = time.perf_counter() - start

lca_time_s = np.nan
if run_lca:
    start = time.perf_counter()
    trails.lca(
        show_progress=show_progress,
        compute_score=True,
        store_inventory=store_inventory,
    )
    lca_time_s = time.perf_counter() - start

add_current_run_to_results(case_name, "adaptive", routing_time_s, lca_time_s)
results[-1]

## 12. Compare the cases

The table below compares graph size, runtime, and scores. Score differences are
computed relative to the fixed-depth 2 case.

In [ ]:
comparison = pd.DataFrame(results).set_index("case")
reference_case = "fixed depth 2"

score_columns = [column for column in comparison.columns if column.startswith("score:")]
for column in score_columns:
    reference_score = float(comparison.loc[reference_case, column])
    comparison[f"difference vs {reference_case}: {column.removeprefix('score: ')}"] = (
        comparison[column].astype(float) - reference_score
    )
    comparison[f"relative difference vs {reference_case}: {column.removeprefix('score: ')}"] = (
        comparison[f"difference vs {reference_case}: {column.removeprefix('score: ')}"]
        / reference_score
        if reference_score != 0
        else np.nan
    )

comparison_path = output_dir / "fixed_depth_vs_adaptive_routing.csv"
comparison.to_csv(comparison_path)
print("Saved:", comparison_path)
comparison

## 13. Plot graph size and runtime

These two plots show whether adaptive routing reached a similar score with fewer
or more explicitly routed nodes, and how much time was spent in routing versus
LCA solving.

In [ ]:
plot_data = comparison.reset_index()

fig_nodes = px.bar(
    plot_data,
    x="case",
    y="nodes",
    color="mode",
    title="Routed graph nodes by case",
)
fig_nodes.update_layout(xaxis_title="Case", yaxis_title="Nodes")
fig_nodes

In [ ]:
fig_time = px.bar(
    plot_data,
    x="case",
    y=["routing_time_s", "lca_time_s"],
    title="Runtime by case",
    barmode="stack",
)
fig_time.update_layout(xaxis_title="Case", yaxis_title="Seconds")
fig_time

## 14. Plot score versus graph size

This plot is useful when you add more fixed depths or relative adaptive cutoff values. It
shows whether additional graph expansion changes the final score meaningfully.

In [ ]:
if score_columns:
    score_column = score_columns[0]
    fig_score = px.scatter(
        plot_data,
        x="nodes",
        y=score_column,
        color="mode",
        text="case",
        title="Total score versus routed graph size",
    )
    fig_score.update_traces(textposition="top center")
    fig_score.update_layout(xaxis_title="Nodes", yaxis_title=score_column.removeprefix("score: "))
    fig_score
else:
    print("No score columns available. Set `run_lca = True` to plot scores.")

## 15. Inspect temporal score curves

The cells below reuse the standard `plot_temporal_scores()` function. They
restore stored score snapshots from two cases so their annual and cumulative
curves can be inspected with the same plotting tool as in Notebook 2.2.

In [ ]:
previous_scores = trails.scores
trails.scores = score_snapshots["fixed depth 2"]

fixed_depth_figures = plot_temporal_scores(
    trails=trails,
    stacked=False,
    legend_top_n=8,
    show_flow_contributions=False,
    title="Fixed depth 2",
    method_label="impact score",
    cumulative=False,
    width=700,
    height=450,
    year_tick=10,
    year_range=(ref_year - 10, ref_year + 80),
    show_cumulative_axis=True,
)

trails.scores = previous_scores
fixed_depth_figures[0]

In [ ]:
previous_scores = trails.scores
trails.scores = score_snapshots["adaptive 1e-2"]

adaptive_figures = plot_temporal_scores(
    trails=trails,
    stacked=False,
    legend_top_n=8,
    show_flow_contributions=False,
    title="Adaptive 1e-2",
    method_label="impact score",
    cumulative=False,
    width=700,
    height=450,
    year_tick=10,
    year_range=(ref_year - 10, ref_year + 80),
    show_cumulative_axis=True,
)

trails.scores = previous_scores
adaptive_figures[0]

## 16. Optional graph visualization

`trails.graph` contains the last routing case that was run. In this notebook that
is the adaptive `1e-2` case unless you rerun an earlier routing cell.

In [ ]:
graph_path = output_dir / "last_routed_graph.html"
plot_temporal_graph(
    trails,
    filename=str(graph_path),
    notebook=False,
)
print("Saved:", graph_path)